# Trustworthy CXR VLM — S0–S3 generation on Kaggle (FP16)

Runs the GPU-heavy generation for the final evaluation (DR-001). Scoring happens locally afterwards
with `src/evaluation/evaluate_stage.py`, so this notebook only produces raw generations.

**Before running**
1. Settings → Accelerator: **GPU T4 x2** (LLaVA-Med 7B in FP16 does not fit one T4). Internet: **On**.
2. Add the IU X-Ray dataset as an input (the one with `indiana_reports.csv`, `indiana_projections.csv`
   and `images/images_normalized/`) and set `IU_DATASET_DIR` below to its path.
3. The prompt used is `vlm.prompt_template_id` in `configs/experiment_config.yaml` of the cloned commit —
   make sure DR-024 has been committed and pushed first.

Every run appends one JSONL line per patient, so a session that times out can be resumed: attach this
notebook's previous output as an input, set `RESUME_DIR`, and run again.

In [ ]:
REPO_URL = "https://github.com/justinthomas11/mts-medical-vlm.git"
REPO_COMMIT = None            # set to the exact commit hash you evaluated locally, for reproducibility
IU_DATASET_DIR = "/kaggle/input/chest-xrays-indiana-university"
RESUME_DIR = None             # e.g. "/kaggle/input/<previous-output>/results" to continue an interrupted run
QUANT = "fp16"
WORK = "/kaggle/working/mts-medical-vlm"

In [ ]:
import os, subprocess, sys
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
if REPO_COMMIT:
    subprocess.run(["git", "-C", WORK, "checkout", REPO_COMMIT], check=True)
os.chdir(WORK)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Same transformers version as local development (DR-022); fp16 needs no bitsandbytes.
!pip install -q "transformers==5.17.0" accelerate
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.device_count(), "GPU(s)",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Link the raw IU X-Ray files where configs/data_config.yaml expects them (data/raw is git-ignored).
os.makedirs("data/raw/images", exist_ok=True)
for src, dst in [("indiana_reports.csv", "data/raw/indiana_reports.csv"),
                 ("indiana_projections.csv", "data/raw/indiana_projections.csv"),
                 ("images/images_normalized", "data/raw/images/images_normalized")]:
    if not os.path.exists(dst):
        os.symlink(os.path.join(IU_DATASET_DIR, src), dst)
print(len(os.listdir("data/raw/images/images_normalized")), "images linked (expected 7,470)")

In [ ]:
# Rebuild data/processed with the fixed seed, then STOP unless the split is identical to the committed one.
subprocess.run([sys.executable, "src/data/split_dataset.py"], check=True)
check = subprocess.run([sys.executable, "src/pipeline/verify_splits.py"])
assert check.returncode == 0, "Split differs from the committed benchmark — do not run any stage."

In [ ]:
# Resume: copy generations from a previous session so finished patients are skipped.
import glob, shutil
if RESUME_DIR:
    for f in glob.glob(os.path.join(RESUME_DIR, "s*", "generations_*.jsonl")):
        dst = os.path.join("results", os.path.relpath(f, RESUME_DIR))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(f, dst)
        print("resumed", dst, sum(1 for _ in open(dst)), "patients")

In [ ]:
def run(*args):
    cmd = [sys.executable, "src/pipeline/run_generation.py", *args, "--quant", QUANT]
    print(" ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

run("--stage", "s0", "--split", "test")                    # S0: greedy, 734 test patients
run("--stage", "s1", "--split", "val", "--samples", "5")   # S1+samples on val: fits the S3 review flag (DR-020)
run("--stage", "s1", "--split", "test", "--samples", "5")  # S1 test; S2 reuses it, S3 adds the samples

In [ ]:
# Package raw generations + run configs for download.
import zipfile
out = "/kaggle/working/phase2_generations.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("results/s*/generations_*.jsonl") + glob.glob("results/s*/run_config_*.json"):
        z.write(f)
        print("added", f)
print(out)

## After downloading

Unzip into the local repo root (it restores `results/s0/…` and `results/s1/…`), then score locally:

```
python src/evaluation/evaluate_stage.py results/s0/generations_test.jsonl
python src/evaluation/evaluate_stage.py results/s1/generations_test.jsonl
python src/evaluation/evaluate_stage.py results/s1/generations_val.jsonl --uncertainty --fit-flagger
python src/evaluation/evaluate_stage.py results/s1/generations_test.jsonl --uncertainty
python src/pipeline/eval_localization.py --split test
python src/evaluation/build_ablation.py --split test
```